## Import / setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

# 將共整合目錄新增到路徑以匯入 get_data 模組
sys.path.insert(0, str(Path.cwd()))

from get_data import (
    create_bybit_session,
    get_all_bybit_usdt_spot_symbols,
    get_all_bybit_perp_symbols,
    fetch_sector_map_with_report,
    download_binance_sector_data,
    explore_downloaded_data,
    preview_market_data,
)

from coint import (
    load_basis_matrices,
    load_sector_map,
    rolling_sector_cointegration_scan,
    walk_forward_basis_backtest,
    build_trading_log,
    summarize_backtest,
)

DATABASE_PATH = Path("data")
CONFIG_PATH = Path("config/api_config.json")  # 選填：用於 API 認證

START_DATE = datetime(2020, 1, 1)
END_DATE = datetime.now()

print(f"數據收集範圍: {START_DATE.date()} 至 {END_DATE.date()}")
print(f"資料庫路徑: {DATABASE_PATH}")
print("數據格式: Parquet")

try:
    bybit_session = create_bybit_session()
    print("Bybit 會話創建成功")
except ImportError as e:
    print(e)
    bybit_session = None

try:
    spot_symbols = get_all_bybit_usdt_spot_symbols()
    perp_symbols = get_all_bybit_perp_symbols()
    print(f"已獲取 {len(spot_symbols)} 個 Bybit 現貨 USDT 符號，範例: {spot_symbols[:5]}")
    print(f"已獲取 {len(perp_symbols)} 個 Bybit 永續 USDT 符號，範例: {perp_symbols[:5]}")
except Exception as e:
    print(f"獲取 Bybit 符號出錯: {e}")
    spot_symbols = []
    perp_symbols = []


# Data

## Fetch Sector Classification


In [ ]:
try:
    fetch_sector_map_with_report(
        base_path=DATABASE_PATH,
        categories_to_process=("spot", "linear"),
        sleep_seconds=0.5,
    )
except ImportError as e:
    print(e)
except Exception as e:
    print(f"獲取行業映射出錯: {e}")

## Download Binance Sector Data


In [ ]:
skip_sectors = ["USD Stablecoin", "Fiat-backed Stablecoin"]

try:
    binance_universe = download_binance_sector_data(
        base_path=DATABASE_PATH,
        start_date=START_DATE,
        end_date=END_DATE,
        skip_sectors=skip_sectors,
        interval="1h",
        sleep_seconds=0.5,
        include_funding_rate=False,
    )
except Exception as e:
    print(f"下載過程中出錯: {e}")

## Data Exploration and Validation


In [ ]:
data_summaries = explore_downloaded_data(DATABASE_PATH)

## Preview Parquet Data


In [ ]:
preview_market_data(DATABASE_PATH)

# Analysis

In [ ]:
# Rolling sector-level futures-spot basis cointegration scan
spot_dir = DATABASE_PATH / "spot"
futures_dir = DATABASE_PATH / "futures"
metadata_path = DATABASE_PATH / "metadata" / "top10_market_sector_map.json"

basis, spot_prices, futures_prices = load_basis_matrices(
    spot_dir=spot_dir,
    futures_dir=futures_dir,
    price_column="close",
    basis_method="log",
    min_obs=500,
)
sector_map = load_sector_map(
    metadata_path,
    skip_sectors=["USD Stablecoin", "Fiat-backed Stablecoin"],
    available_symbols=basis.columns,
)

print(f"basis 矩陣: {basis.shape[0]} 筆時間資料 x {basis.shape[1]} 個幣種")
print(f"可分析板塊數: {len(sector_map)}")

rolling_basis_pairs = rolling_sector_cointegration_scan(
    prices=basis,
    sector_map=sector_map,
    formation_window="60D",
    trading_window="7D",
    step="7D",
    max_pvalue=0.05,
    max_spread_adf_pvalue=0.05,
    min_obs=1000,
    top_n_per_sector=3,
)

rolling_basis_backtests, rolling_basis_summaries = walk_forward_basis_backtest(
    basis=basis,
    spot_prices=spot_prices,
    futures_prices=futures_prices,
    rolling_pairs=rolling_basis_pairs,
    max_pairs_per_window=10,
    entry_z=2.0,
    exit_z=0.5,
    fee_rate=0.0004,
)

basis_trading_log = build_trading_log(rolling_basis_backtests)

print(f"rolling basis pair 數量: {len(rolling_basis_pairs)}")
print(f"回測 pair-window 數量: {len(rolling_basis_summaries)}")
print(f"交易筆數: {len(basis_trading_log)}")

if rolling_basis_summaries.empty:
    print("沒有 rolling window 產生可回測的 basis 配對，請放寬 p-value、縮短 min_obs，或確認期現資料期間足夠。")
else:
    display(rolling_basis_pairs.head(20))
    display(rolling_basis_summaries.sort_values(["window_id", "sharpe"], ascending=[True, False]).head(20))
    basis_trading_log.head(50)
